# Esercitazione Realsense - Mask 

## Imports and Setup

In [ ]:
import numpy as np
import glob
import open3d as o3d
import matplotlib.pyplot as plt
import os
import re
import copy
from matplotlib import cm

In [ ]:
path_117322070663 = "./mask/117322070663_*.ply"
path_108322073284 = "./mask/108322073284_*.ply"

path_117322070663_setup_2 = "./mask_2/117322070663_*.ply"
path_108322073284_setup_2 = "./mask_2/108322073284_*.ply"

In [ ]:
camera_params = {
    "117322070663": {
        "K": np.array([[591.2588416325026, 0.0, 303.96766080041255], [0.0, 582.3094248920196, 253.6549343618132], [0.0, 0.0, 1.0]]),
        "D": np.array([0.0857423535202526, 0.15899390827577137, 0.0027644426081692413, -0.014441335296417794, -1.3507539704153515])
    },
    "108322073284": {
        "K": np.array([[581.4988972720037, 0.0, 339.4397798655603], [0.0, 582.8600048067065, 282.73657769938495], [0.0, 0.0, 1.0]]),
        "D": np.array([-0.24057957220615858, 4.2767106826695604, 0.024711451849462666, 0.013758143071840522, -20.808592661502967])
    }
}

camera_params_setup_2 = {
    "117322070663": {
        "K": np.array([[573.71684236, 0.0, 323.77041426], [0.0, 568.08159744, 291.30687122], [0.0, 0.0, 1.0]]),
        "D": np.array([-0.050235, 0.8596034, 0.02748887, 0.00546327, -3.2747901])
    },
    "108322073284": {
        "K": np.array([[582.88962458, 0.0, 319.87188926], [0.0, 585.14884733, 259.19615629], [0.0, 0.0, 1.0]]),
        "D": np.array([-0.05514187, 1.29116953, 0.0161248, -0.01959811, -4.33660378])
    }
}

R = np.array([
    [-0.15352943086938806, 0.47550867465054586, -0.8662102598035901],
    [-0.52601996, 0.7027474, 0.47900845],
    [0.83649968, 0.52918578, 0.14223465]
], dtype=float)

t = np.array([0.27529726787218745, -0.18074483, 0.22664823], dtype=float).reshape(3, 1)

R_setup_2 = np.array([
    [0.5351167552416705, 0.6482807864565784, -0.541647560846411],
    [-0.64783462, 0.72641869, 0.229404],
    [0.54218111, 0.22814012, 0.80869755]
], dtype=float)

t_setup_2 = np.array([0.3746002611726339, -0.07860003, 0.10068186], dtype=float).reshape(3, 1)

In [ ]:
def _idx_from_name(path):
    m = re.search(r'_(\d+)\.ply$', os.path.basename(path))
    return int(m.group(1)) if m else None

def _files_by_idx(pattern):
    files = glob.glob(pattern)
    out = {}
    for f in files:
        idx = _idx_from_name(f)
        if idx is not None:
            out[idx] = f
    return out

def _make_T(Rm, tv):
    Tm = np.eye(4)
    Tm[:3, :3] = Rm
    Tm[:3, 3] = tv.reshape(3)
    return Tm

In [ ]:
def pointcloud_to_depth_and_color_image(
    pcd, K, width=640, height=480, flip_y=False, flip_z=False, depth_positive='abs'
 ):
    pts = np.asarray(pcd.points).copy()
    cols = np.asarray(pcd.colors).copy() if len(pcd.colors) > 0 else None

    if pts.shape[0] == 0:
        return np.full((height, width), np.nan), np.zeros((height, width, 3), dtype=np.uint8)

    if flip_y:
        pts[:, 1] *= -1.0
    if flip_z:
        pts[:, 2] *= -1.0

    X, Y, Z = pts[:, 0], pts[:, 1], pts[:, 2]

    Z_proj = Z

    valid_z = np.isfinite(Z_proj) & (np.abs(Z_proj) > 1e-9)
    X, Y, Z_proj = X[valid_z], Y[valid_z], Z_proj[valid_z]

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]

    u = np.round((fx * X) / Z_proj + cx).astype(np.int32)
    v = np.round((fy * Y) / Z_proj + cy).astype(np.int32)

    in_img = (u >= 0) & (u < width) & (v >= 0) & (v < height)
    u, v, Z_proj = u[in_img], v[in_img], Z_proj[in_img]

    if cols is not None and cols.shape[0] == pts.shape[0]:
        cols = cols[valid_z][in_img]
        cols = np.clip(np.round(cols * 255.0), 0, 255).astype(np.uint8)
    else:
        cols = np.full((u.shape[0], 3), 127, dtype=np.uint8)

    depth_map = np.full((height, width), np.nan, dtype=np.float32)
    color_img = np.zeros((height, width, 3), dtype=np.uint8)

    flat_idx = v * width + u
    order = np.argsort(Z_proj)
    flat_sorted = flat_idx[order]
    keep = np.ones(flat_sorted.shape[0], dtype=bool)
    keep[1:] = flat_sorted[1:] != flat_sorted[:-1]
    chosen = order[keep]

    u_c, v_c = u[chosen], v[chosen]
    depth_map[v_c, u_c] = Z_proj[chosen]
    color_img[v_c, u_c] = cols[chosen]

    return depth_map, color_img

def show_depth_and_color_from_ply(ply_path, K, title_prefix, flip_y=False, flip_z=False):
    pcd = o3d.io.read_point_cloud(ply_path)
    depth_map, color_img = pointcloud_to_depth_and_color_image(
        pcd, K, width=640, height=480, flip_y=flip_y, flip_z=flip_z, depth_positive='abs'
    )

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(depth_map, cmap='jet')
    plt.title(f'{title_prefix} - Depth map (Z)')
    plt.colorbar(label='Depth [m]')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(color_img)
    plt.title(f'{title_prefix} - Color from point cloud')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def _files_by_idx(pattern):
    out = {}
    for f in glob.glob(pattern):
        name = f.rsplit("/", 1)[-1]
        try:
            idx = int(name.rsplit("_", 1)[-1].split(".ply")[0])
            out[idx] = f
        except (ValueError, IndexError):
            continue
    return out

In [ ]:
np.set_printoptions(precision=4, suppress=True, linewidth=100)

print("\n" + "="*60)
print("  DETERMINANTS AND ORTHOGONALITY CHECKS")
print("="*60)
print(f"\n  det(R)         = {np.linalg.det(R):10.6f}")
print(f"  det(R_setup_2) = {np.linalg.det(R_setup_2):10.6f}")
print("\n" + "-"*60)
print("  R @ R.T (should be ≈ I):")
print("-"*60)
print(R @ R.T)
print("\n" + "-"*60)
print("  R_setup_2 @ R_setup_2.T (should be ≈ I):")
print("-"*60)
print(R_setup_2 @ R_setup_2.T)
print("\n" + "="*60 + "\n")

## Setup 1

In [ ]:
m117 = _files_by_idx(path_117322070663)
m108 = _files_by_idx(path_108322073284)
common = sorted(set(m117.keys()) & set(m108.keys()))

if not common:
    print('No frame in common found.')
else:
    frame_idx = common[0]
    show_depth_and_color_from_ply(
        m117[frame_idx], camera_params['117322070663']['K'], f'Camera 117 - frame {frame_idx}'
    )
    show_depth_and_color_from_ply(
        m108[frame_idx], camera_params['108322073284']['K'], f'Camera 108 - frame {frame_idx}'
    )

In [ ]:
def camera_to_image(K, ptc, name="Depth Map"):
    X = ptc[:, 0]
    Y = ptc[:, 1]
    Z = ptc[:, 2]

    Z_abs = Z

    fx, fy = K[0, 0], K[1, 1]
    cx, cy = K[0, 2], K[1, 2]
    
    u = ((fx * X) / Z_abs) + cx
    v = ((fy * Y) / Z_abs) + cy

    depth_map = np.zeros((480, 640))
    u_int = np.round(u).astype(int)
    v_int = np.round(v).astype(int)
    
    mask = (u_int >= 0) & (u_int < 640) & (v_int >= 0) & (v_int < 480)
    
    depth_map[v_int[mask], u_int[mask]] = Z_abs[mask]
    
    depth_map[depth_map == 0] = np.nan

    plt.figure(figsize=(8, 6))
    img = plt.imshow(depth_map, cmap='jet')
    plt.colorbar(img, label='Distanza (m)')
    plt.title(name)
    plt.show()

    return u, v

In [ ]:
camera_117322070663 = []

for element in glob.glob(path_117322070663):
    pcd = o3d.io.read_point_cloud(element)
    points = np.asarray(pcd.points)

    camera_117322070663.append(points)

In [ ]:
camera_108322073284 = []

for element in glob.glob(path_108322073284):
    pcd = o3d.io.read_point_cloud(element)
    points = np.asarray(pcd.points)

    camera_108322073284.append(points)

In [ ]:
# for element in camera_117322070663:
    # u, v = camera_to_image(camera_params["117322070663"]["K"], element)

In [ ]:
# for element in camera_108322073284:
    # u, v = camera_to_image(camera_params["108322073284"]["K"], element)

In [ ]:
def compare_and_show(frame_position=0):
    m117 = _files_by_idx(path_117322070663)
    m108 = _files_by_idx(path_108322073284)
    common = sorted(set(m117.keys()) & set(m108.keys()))
    
    if not common:
        print("No frame found.")
        return
        
    idx = common[max(0, min(frame_position, len(common)-1))]

    tgt = o3d.io.read_point_cloud(m117[idx])
    src = o3d.io.read_point_cloud(m108[idx])

    T_cv = _make_T(R, t)
    F = np.diag([1, -1, -1, 1]).astype(float)
    T_init = F @ T_cv @ F

    src.transform(T_init)

    # ritaglio per rimuovere la stanza e mantenere solo la maschera (e dintorni)
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(-0.15, -0.15, -0.34),
        max_bound=( 0.15,  0.15, 0.5)
    )
    
    tgt = tgt.crop(bbox)
    src = src.crop(bbox)

    print(f":: Target: {len(tgt.points)} | Source: {len(src.points)}")
    if len(tgt.points) < 100 or len(src.points) < 100:
        print("ERROR: empty bounding box after cropping! Adjust the bounding box or check the initial alignment.")
        return

    voxel_size = 0.004
    tgt = tgt.voxel_down_sample(voxel_size)
    src = src.voxel_down_sample(voxel_size)

    tgt, _ = tgt.remove_statistical_outlier(nb_neighbors=30, std_ratio=2.0)
    src, _ = src.remove_statistical_outlier(nb_neighbors=30, std_ratio=2.0)

    search_radius = 0.02
    tgt.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=search_radius, max_nn=30))
    tgt.orient_normals_towards_camera_location(np.array([0., 0., 0.]))
    
    src.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=search_radius, max_nn=30))
    src.orient_normals_towards_camera_location(np.array([0., 0., 0.]))

    print(":: 1: Coarse ICP")
    coarse_distance = 0.05
    reg_coarse = o3d.pipelines.registration.registration_icp(
        src, tgt, coarse_distance, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=100))

    print(":: 2: Fine Colored ICP")
    fine_distance = 0.015
    reg_icp = o3d.pipelines.registration.registration_colored_icp(
        src, tgt, fine_distance, reg_coarse.transformation,
        o3d.pipelines.registration.TransformationEstimationForColoredICP(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            relative_fitness=1e-6, 
            relative_rmse=1e-6, 
            max_iteration=150))

    src_final = copy.deepcopy(src)
    src_final.transform(reg_icp.transformation)

    flip_view = np.diag([1, -1, -1, 1]).astype(float)
    src_final.transform(flip_view)
    
    tgt_final = copy.deepcopy(tgt)
    tgt_final.transform(flip_view)

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="ICP Colored")

    vis.add_geometry(tgt_final)
    vis.add_geometry(src_final)

    view_ctl = vis.get_view_control()

    cam_param = o3d.io.read_pinhole_camera_parameters("camera_view.json")
    
    view_ctl.convert_from_pinhole_camera_parameters(cam_param)

    print(":: Visualizing result...")
    vis.run()
    vis.destroy_window()
    
if not hasattr(o3d, "visualization"):
    import open3d.cpu.pybind as o3d_cpu
    o3d.visualization = o3d_cpu.visualization

compare_and_show()

## Setup 2
It uses the same code as the previous setup, but there are some difference in the ICP usage.

In [ ]:
m117_s2 = _files_by_idx(path_117322070663_setup_2)
m108_s2 = _files_by_idx(path_108322073284_setup_2)
common_s2 = sorted(set(m117_s2.keys()) & set(m108_s2.keys()))

if not common_s2:
    print('No frame in common found for Setup 2.')
else:
    frame_idx_s2 = common_s2[0]
    
    flip_y_attivo = True
    flip_z_attivo = True 
    
    show_depth_and_color_from_ply(
        m117_s2[frame_idx_s2], 
        camera_params_setup_2['117322070663']['K'], 
        f'Setup 2 - Camera 117',
        flip_y=flip_y_attivo,
        flip_z=flip_z_attivo
    )
    
    show_depth_and_color_from_ply(
        m108_s2[frame_idx_s2], 
        camera_params_setup_2['108322073284']['K'], 
        f'Setup 2 - Camera 108',
        flip_y=flip_y_attivo,
        flip_z=flip_z_attivo
    )

In [ ]:
def debug_initial_alignment():
    m117 = _files_by_idx(path_117322070663_setup_2)
    m108 = _files_by_idx(path_108322073284_setup_2)
    idx = list(m117.keys())[0]

    tgt = o3d.io.read_point_cloud(m117[idx])
    src = o3d.io.read_point_cloud(m108[idx])

    tgt = tgt.voxel_down_sample(0.005)
    src = src.voxel_down_sample(0.005)

    T_cv = _make_T(R_setup_2, t_setup_2)
    F = np.diag([1, -1, -1, 1]).astype(float)
    T_init = F @ T_cv @ F
    src.transform(T_init)
    
    shift_x = 0.00  
    shift_y = 0.05
    shift_z = 0.025

    src.translate((shift_x, shift_y, shift_z))

    tgt.paint_uniform_color([0.1, 0.7, 1.0])
    src.paint_uniform_color([1.0, 0.2, 0.2])

    flip_view = np.diag([1, -1, -1, 1]).astype(float)
    tgt.transform(flip_view)
    src.transform(flip_view)

    frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.1)

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="DEBUG")
    vis.add_geometry(tgt)
    vis.add_geometry(src)
    vis.add_geometry(frame)

    view_ctl = vis.get_view_control()
    try:
        cam_param = o3d.io.read_pinhole_camera_parameters("camera_view_2.json")
        view_ctl.convert_from_pinhole_camera_parameters(cam_param)
    except Exception as e:
        print("Warning: could not load camera parameters. Using default view.")
        
        view_ctl.set_zoom(0.7)

    vis.run()
    vis.destroy_window()

debug_initial_alignment()

In [ ]:
import open3d as o3d
import numpy as np
import copy

def compare_and_show_setup_2(frame_position=0):
    m117 = _files_by_idx(path_117322070663_setup_2)
    m108 = _files_by_idx(path_108322073284_setup_2)
    common = sorted(set(m117.keys()) & set(m108.keys()))
    
    if not common:
        print("No frame found in mask_2.")
        return
    
    idx = common[max(0, min(frame_position, len(common) - 1))]

    tgt = o3d.io.read_point_cloud(m117[idx])
    src = o3d.io.read_point_cloud(m108[idx])

    T_cv = _make_T(R_setup_2, t_setup_2)
    F = np.diag([1, -1, -1, 1]).astype(float)
    src.transform(F @ T_cv @ F)

    # src.translate((0.00, 0.05, 0.025))

    tgt_full = tgt.voxel_down_sample(0.004)
    src_full = src.voxel_down_sample(0.004)

    bbox_face = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=(-0.10, -0.15, -0.65),
        max_bound=( 0.10,  0.05, -0.15)
    )

    tgt_face = tgt_full.crop(bbox_face)
    src_face = src_full.crop(bbox_face)

    tgt_face, _ = tgt_face.remove_statistical_outlier(nb_neighbors=30, std_ratio=2.0)
    src_face, _ = src_face.remove_statistical_outlier(nb_neighbors=30, std_ratio=2.0)

    search_radius = 0.02
    tgt_face.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=search_radius, max_nn=30))
    tgt_face.orient_normals_towards_camera_location(np.array([0.0, 0.0, 0.0]))
    
    src_face.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=search_radius, max_nn=30))
    src_face.orient_normals_towards_camera_location(np.array([0.0, 0.0, 0.0]))

    print(":: Colored ICP")
    fine_distance = 0.02 
    
    reg_icp = o3d.pipelines.registration.registration_colored_icp(
        src_face, tgt_face, fine_distance, np.eye(4),
        o3d.pipelines.registration.TransformationEstimationForColoredICP(),
        o3d.pipelines.registration.ICPConvergenceCriteria(
            relative_fitness=1e-6, relative_rmse=1e-6, max_iteration=200)
    )
    print(f":: ICP -> Fitness: {reg_icp.fitness:.4f} | RMSE: {reg_icp.inlier_rmse:.4f}")

    src_final = copy.deepcopy(src_full)
    src_final.transform(reg_icp.transformation)

    flip_view = np.diag([1, -1, -1, 1]).astype(float)
    src_final.transform(flip_view)
    
    tgt_final = copy.deepcopy(tgt_full)
    tgt_final.transform(flip_view)

    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name="ICP Colored - setup_2")
    vis.add_geometry(tgt_final)
    vis.add_geometry(src_final)

    try:
        view_ctl = vis.get_view_control()
        cam_param = o3d.io.read_pinhole_camera_parameters("camera_view_2.json")
        view_ctl.convert_from_pinhole_camera_parameters(cam_param)
    except Exception:
        view_ctl.set_lookat(tgt_final.get_center())

    vis.run()
    vis.destroy_window()

compare_and_show_setup_2()

# Wenglor Profile Sensor

In [ ]:
x_array = []
y_array = []
z_array = []

with open("prova1.txt", "r") as f1:
    for i, line in enumerate(f1):
        line = line.strip()
        if not line:
            continue

        if i == 0 and line.upper().startswith("X"):
            continue

        line = line.replace("\\t", "\t")
        parts = [p for p in re.split(r"\t+|\s+", line) if p]

        if len(parts) < 3:
            continue

        x = float(parts[0].replace(",", "."))
        y = float(parts[1].replace(",", "."))
        z = float(parts[2].replace(",", "."))

        x_array.append(x)
        y_array.append(y)
        z_array.append(z)

x = np.asarray(x_array, dtype=float)
y = np.asarray(y_array, dtype=float)
z = np.asarray(z_array, dtype=float)

soglia_z_minima = 120.0 

mask = z > soglia_z_minima
x = x[mask]
y = y[mask]
z = z[mask]

print(f"Points after filtering: {len(x)}")

import open3d as o3d

punti_xyz = np.column_stack((x, y, z))

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(punti_xyz)

pcd.paint_uniform_color([0, 0.5, 0])

o3d.visualization.draw_geometries([pcd], window_name="Wrenglor Point Cloud", width=800, height=600)